# Task 2


## 1. Install and import libraries

In [ ]:


!pip install -q "transformers==4.46.3" "sentence-transformers==3.3.1" accelerate faiss-cpu pypdf


In [22]:

import torch
import pandas as pd
import transformers
import sentence_transformers
import faiss

print("Python environment ready!")
print("Pandas:", pd.__version__)
print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("FAISS:", faiss.__version__)
print("GPU:", torch.cuda.is_available())


Python environment ready!
Pandas: 3.1.1
Transformers: 5.23.0
Sentence Transformers: 6.3.0
FAISS: 1.16.0
GPU: False


In [23]:

import transformers
import pandas as pd

print("Transformers:", transformers.__version__)
print("Pandas:", pd.__version__)

from transformers import pipeline

print("Pipeline imported successfully!")


Transformers: 5.23.0
Pandas: 3.1.1
Pipeline imported successfully!


In [24]:
import os, re, json, textwrap, time
import numpy as np
import pandas as pd
import torch
import faiss

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

SEED = 42
np.random.seed(SEED)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.14.1+cpu
CUDA available: False


## 2. Download the official NIST AI RMF document

This notebook uses **NIST AI 100-1, AI Risk Management Framework (AI RMF 1.0)**.

Official NIST publication page: https://www.nist.gov/publications/artificial-intelligence-risk-management-framework-ai-rmf-10


In [25]:
PDF_URL = "https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.100-1.pdf"
PDF_PATH = "NIST_AI_RMF_1.0.pdf"

if not os.path.exists(PDF_PATH):
    import requests
    response = requests.get(PDF_URL, timeout=60)
    response.raise_for_status()
    with open(PDF_PATH, "wb") as f:
        f.write(response.content)

reader = PdfReader(PDF_PATH)
pages = []
for page_num, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text() or ""
    if page_text.strip():
        pages.append({"page": page_num, "text": page_text})

print("PDF pages:", len(reader.pages))
print("Pages with extractable text:", len(pages))
print("First page preview:")
print(pages[0]["text"][:1000] if pages else "No text extracted")


PDF pages: 48
Pages with extractable text: 48
First page preview:
NIST AI 100-1
Artificial Intelligence Risk Management
Framework (AI RMF 1.0)



## 3. Prepare text and create token-based chunks

We compare **300-token** and **600-token** chunks. The tokenizer below is from the same embedding model used later. Each chunk retains its original PDF page number so retrieved evidence can be traced back to the document.


In [26]:
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME)

chunk_tokenizer = embedder.tokenizer

def clean_text(text):
    text = text.replace("\\x00", " ")
    text = re.sub(r"\\s+", " ", text)
    return text.strip()

def make_chunks(page_records, chunk_size, overlap=50):
    chunks = []
    for record in page_records:
        text = clean_text(record["text"])
        token_ids = chunk_tokenizer.encode(text, add_special_tokens=False)
        start = 0
        while start < len(token_ids):
            end = min(start + chunk_size, len(token_ids))
            chunk_ids = token_ids[start:end]
            chunk_text = chunk_tokenizer.decode(chunk_ids, skip_special_tokens=True).strip()
            if chunk_text:
                chunks.append({
                    "text": chunk_text,
                    "page": record["page"],
                    "start_token": start,
                    "end_token": end
                })
            if end == len(token_ids):
                break
            start = max(end - overlap, start + 1)
    return chunks

chunks_300 = make_chunks(pages, chunk_size=300, overlap=50)
chunks_600 = make_chunks(pages, chunk_size=600, overlap=50)

print("300-token chunks:", len(chunks_300))
print("600-token chunks:", len(chunks_600))
print("\\nExample 300-token chunk:")
print(chunks_300[0]["text"][:1000])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (327 > 256). Running this sequence through the model will result in indexing errors


300-token chunks: 97
600-token chunks: 51
\nExample 300-token chunk:
nist ai 100 - 1 artificial intelligence risk management framework ( ai rmf 1. 0 )


## 4. Embed chunks and build FAISS indexes

Embeddings are normalized and searched with inner product, which is equivalent to cosine similarity for normalized vectors.


In [27]:
def build_faiss_index(chunks, embedder, batch_size=32):
    texts = [c["text"] for c in chunks]
    vectors = embedder.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    ).astype("float32")
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return index, vectors

index_300, vectors_300 = build_faiss_index(chunks_300, embedder)
index_600, vectors_600 = build_faiss_index(chunks_600, embedder)

print("FAISS 300 index vectors:", index_300.ntotal)
print("FAISS 600 index vectors:", index_600.ntotal)


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

FAISS 300 index vectors: 97
FAISS 600 index vectors: 51


## 5. Define 15 evaluation questions

These questions cover the framework's purpose, trustworthiness characteristics, lifecycle, and the four core functions. The answers will be generated from retrieved passages, not from a manually supplied answer key.


In [28]:
questions = [
    "What is the main purpose of the NIST AI Risk Management Framework?",
    "Is the NIST AI RMF mandatory for organizations?",
    "Which organizations or people are the intended audience of the AI RMF?",
    "What are the four core functions of the AI RMF?",
    "What does the GOVERN function address?",
    "What is the purpose of the MAP function?",
    "What does the MEASURE function help organizations do?",
    "What is the purpose of the MANAGE function?",
    "What does it mean for an AI system to be valid and reliable?",
    "Which trustworthiness characteristics are identified in the AI RMF?",
    "Why is AI risk management described as socio-technical?",
    "At which stages of the AI lifecycle should trustworthiness be considered?",
    "What is an AI system according to the framework?",
    "Why can AI risks be difficult to identify and manage?",
    "How does the AI RMF help organizations address potential harms from AI systems?"
]
assert len(questions) == 15
print("Evaluation questions:", len(questions))


Evaluation questions: 15


## 6. Retrieve relevant passages for each question

In [29]:
def retrieve(question, chunks, index, top_k=3):
    q_vec = embedder.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    scores, ids = index.search(q_vec, top_k)
    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        item = dict(chunks[int(idx)])
        item["score"] = float(score)
        results.append(item)
    return results

# Quick retrieval check
sample_hits = retrieve(questions[0], chunks_300, index_300, top_k=3)
for i, hit in enumerate(sample_hits, 1):
    print(f"\\n[{i}] Page {hit['page']} | similarity={hit['score']:.4f}")
    print(hit["text"][:700])


\n[1] Page 1 | similarity=0.7687
nist ai 100 - 1 artificial intelligence risk management framework ( ai rmf 1. 0 )
\n[2] Page 2 | similarity=0.7594
nist ai 100 - 1 artificial intelligence risk management framework ( ai rmf 1. 0 ) this publication is available free of charge from : https : / / doi. org / 10. 6028 / nist. ai. 100 - 1 january 2023 u. s. department of commerce gina m. raimondo, secretary national institute of standards and technology laurie e. locascio, nist director and under secretary of commerce for standards and technology
\n[3] Page 6 | similarity=0.7273
nist ai 100 - 1 ai rmf 1. 0 executive summary artificial intelligence ( ai ) technologies have significant potential to transform society and people ’ s lives – from commerce and health to transportation and cybersecurity to the envi - ronment and our planet. ai technologies can drive inclusive economic growth and support scientific advancements that improve the conditions of our world. ai technologies, how - ever, al

## 7. Generate answers using retrieved context

The answer model receives only the retrieved passages and the question. If the evidence is insufficient, it is instructed to say so rather than inventing an answer. The source passages and page numbers are saved with every response.


In [30]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

GEN_MODEL_NAME = "google/flan-t5-base"

device = "cuda" if torch.cuda.is_available() else "cpu"

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)

gen_model = AutoModelForSeq2SeqLM.from_pretrained(
    GEN_MODEL_NAME
).to(device)

print("Generation model loaded successfully!")
print("Device:", device)


def answer_with_context(question, hits, max_new_tokens=160):

    context_parts = []

    for i, hit in enumerate(hits, 1):
        context_parts.append(
            f"[Source {i}, PDF page {hit['page']}] {hit['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = (
        "Answer the question using only the source passages below. "
        "If the passages do not provide enough information, say so. "
        "Do not add unsupported facts.\n\n"
        f"Source passages:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )

    inputs = gen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    return gen_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

print("Answer generation function ready!")



Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generation model loaded successfully!
Device: cpu
Answer generation function ready!


## 8. Run the 15-question evaluation for both chunk sizes

For each question, the notebook records the generated answer, retrieved passages, page numbers, and similarity scores. This makes it possible to inspect whether the answer is supported by the retrieved evidence.


In [15]:
def evaluate_chunk_size(chunk_label, chunks, index, top_k=3):
    rows = []
    for q_id, question in enumerate(questions, start=1):
        hits = retrieve(question, chunks, index, top_k=top_k)
        answer = answer_with_context(question, hits)
        rows.append({
            "question_id": q_id,
            "chunk_size": chunk_label,
            "question": question,
            "answer": answer,
            "source_pages": ", ".join(str(h["page"]) for h in hits),
            "retrieved_passages": "\\n\\n---\\n\\n".join(
                f"(Page {h['page']}; score {h['score']:.4f})\\n{h['text']}"
                for h in hits
            ),
            "similarity_scores": [round(h["score"], 4) for h in hits]
        })
        print(f"Completed {chunk_label}-token question {q_id}/15")
    return rows

start_time = time.time()
results_300 = evaluate_chunk_size(300, chunks_300, index_300, top_k=3)
results_600 = evaluate_chunk_size(600, chunks_600, index_600, top_k=3)
print("Evaluation minutes:", round((time.time() - start_time) / 60, 2))


Completed 300-token question 1/15
Completed 300-token question 2/15
Completed 300-token question 3/15
Completed 300-token question 4/15
Completed 300-token question 5/15
Completed 300-token question 6/15
Completed 300-token question 7/15
Completed 300-token question 8/15
Completed 300-token question 9/15
Completed 300-token question 10/15
Completed 300-token question 11/15
Completed 300-token question 12/15
Completed 300-token question 13/15
Completed 300-token question 14/15
Completed 300-token question 15/15
Completed 600-token question 1/15
Completed 600-token question 2/15
Completed 600-token question 3/15
Completed 600-token question 4/15
Completed 600-token question 5/15
Completed 600-token question 6/15
Completed 600-token question 7/15
Completed 600-token question 8/15
Completed 600-token question 9/15
Completed 600-token question 10/15
Completed 600-token question 11/15
Completed 600-token question 12/15
Completed 600-token question 13/15
Completed 600-token question 14/15
Com

## 9. Inspect answers and evidence

Review all 30 rows. Check whether each answer is accurate and whether the cited retrieved passage actually supports it. Do not report a score until you have inspected the answers.


In [19]:

results_df = pd.DataFrame(results_300 + results_600)

print("Results DataFrame created!")
print("Total evaluation records:", len(results_df))


Results DataFrame created!
Total evaluation records: 30


In [20]:
# Display full retrieved evidence for one question from each configuration.
for chunk_label in [300,  600]:
    row = results_df[(results_df["question_id"] == 1) & (results_df["chunk_size"] == chunk_label)].iloc[0]
    print(f"\\n{'='*80}\\nCHUNK SIZE: {chunk_label}\\nQUESTION: {row['question']}\\nANSWER: {row['answer']}\\nSOURCE PAGES: {row['source_pages']}\\n")
    print(row["retrieved_passages"])


\n================================================================================\nCHUNK SIZE: 300\nQUESTION: What is the main purpose of the NIST AI Risk Management Framework?\nANSWER: The ai rmf refers to an ai system as an engineered or machine - based system that can, for a given set of objectives, generate outputs such as predictions, recommenda - tions, or decisions influencing real or virtual environments.\nSOURCE PAGES: 1, 2, 6\n
(Page 1; score 0.7687)\nnist ai 100 - 1 artificial intelligence risk management framework ( ai rmf 1. 0 )\n\n---\n\n(Page 2; score 0.7594)\nnist ai 100 - 1 artificial intelligence risk management framework ( ai rmf 1. 0 ) this publication is available free of charge from : https : / / doi. org / 10. 6028 / nist. ai. 100 - 1 january 2023 u. s. department of commerce gina m. raimondo, secretary national institute of standards and technology laurie e. locascio, nist director and under secretary of commerce for standards and technology\n\n---\n\n(Page 6; 

## 10. Save results and artifacts

Download the CSV and JSON files from the Colab Files panel. Keep the notebook and the NIST source PDF with your submission. The JSON file preserves full retrieved passages for evidence review.


In [21]:
results_df.to_csv("task2_rag_results.csv", index=False)

with open("task2_rag_results.json", "w", encoding="utf-8") as f:
    json.dump(results_300 + results_600, f, indent=2, ensure_ascii=False)

summary = {
    "source_document": "NIST AI Risk Management Framework (AI RMF 1.0), NIST AI 100-1",
    "embedding_model": EMBED_MODEL_NAME,
    "generation_model": GEN_MODEL_NAME,
    "chunk_sizes": [300, 600],
    "overlap_tokens": 50,
    "top_k": 3,
    "questions_per_configuration": len(questions),
    "chunks_300": len(chunks_300),
    "chunks_600": len(chunks_600),
    "faiss_vectors_300": int(index_300.ntotal),
    "faiss_vectors_600": int(index_600.ntotal),
    "note": "Generated answers and retrieval evidence must be manually reviewed before reporting quality metrics."
}
with open("task2_rag_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved: task2_rag_results.csv")
print("Saved: task2_rag_results.json")
print("Saved: task2_rag_summary.json")


Saved: task2_rag_results.csv
Saved: task2_rag_results.json
Saved: task2_rag_summary.json


## 11. Conclusion and Limitations

This project implemented a retrieval-augmented question-answering system using the NIST AI Risk Management Framework (AI RMF 1.0). The document was extracted from PDF, divided into overlapping 300-token and 600-token chunks, embedded with Sentence Transformers (all-MiniLM-L6-v2), and indexed using FAISS. Answers were generated with google/flan-t5-base using the top three retrieved passages.

The comparison below is based on the outputs generated by this notebook. Review the retrieved passages and answers before drawing conclusions about which chunk size performed better.

Limitations: The evaluation uses one source document and 15 questions. Retrieval is based on embedding similarity, and answer quality requires manual review of whether each response is accurate and supported by its retrieved passages. Results may differ with other documents, questions, chunking choices, and model settings.